# Tarea 1 — Web Semántica y Datos Abiertos (MTI)
## Knowledge Graph: Producción Animal (RDF + ShEx)

**Qué hace este notebook:**
- Carga datos **privados** (CSV) de Producción Animal y los transforma a **RDF/Turtle**.
- Carga datos **abiertos** (CSV, no-RDF) de comunas/regiones y los transforma a **RDF/Turtle**.
- Une ambos grafos (merge) y valida con **Shape Expressions (ShEx)**.
- Incluye un ejemplo **inválido** para demostrar que la validación detecta errores.

> Archivos de salida: `private.ttl`, `open.ttl`, `merged.ttl`, `schema.shex`, `invalid_examples.ttl`


In [ ]:
# (1) Instalar dependencias
!pip -q install rdflib pyshex pandas


## (2) Subir archivos CSV
Sube estos 4 archivos:
- `farms_private.csv`
- `animals_private.csv`
- `weights_private.csv`
- `comunas_open.csv`

Si no los tienes, puedes usar los ejemplos que vienen en el ZIP que generamos.

In [ ]:
from google.colab import files
uploaded = files.upload()
print('Archivos subidos:', list(uploaded.keys()))


## (3) Transformación: datos privados → RDF (Turtle)
Genera `private.ttl` con granjas, animales y registros de pesaje.

In [ ]:
import pandas as pd
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, XSD

# Namespace propio (puedes cambiarlo por uno real si quieres)
EX = Namespace('https://kg.example.cl/animal/')
SCHEMA = Namespace('https://schema.org/')

g_priv = Graph()
g_priv.bind('ex', EX)
g_priv.bind('schema', SCHEMA)

farms = pd.read_csv('farms_private.csv')
animals = pd.read_csv('animals_private.csv')
weights = pd.read_csv('weights_private.csv')

# Farms
for _, f in farms.iterrows():
    farm_uri = URIRef(EX[f"farm/{f['farm_id']}"])
    comuna_uri = URIRef(EX[f"comuna/{int(f['comuna_code'])}"])

    g_priv.add((farm_uri, RDF.type, EX.Farm))
    g_priv.add((farm_uri, SCHEMA.name, Literal(f['name'])))
    g_priv.add((farm_uri, EX.locatedInComuna, comuna_uri))

# Animals
for _, a in animals.iterrows():
    animal_uri = URIRef(EX[f"animal/{a['animal_id']}"])
    farm_uri = URIRef(EX[f"farm/{a['farm_id']}"])

    g_priv.add((animal_uri, RDF.type, EX.Animal))
    g_priv.add((animal_uri, EX.species, Literal(a['species'])))
    g_priv.add((animal_uri, EX.birthDate, Literal(a['birth_date'], datatype=XSD.date)))
    g_priv.add((animal_uri, EX.raisedInFarm, farm_uri))

# Weight records
for _, w in weights.iterrows():
    rec_uri = URIRef(EX[f"weightRecord/{w['record_id']}"])
    animal_uri = URIRef(EX[f"animal/{w['animal_id']}"])

    g_priv.add((rec_uri, RDF.type, EX.WeightRecord))
    g_priv.add((rec_uri, EX.forAnimal, animal_uri))
    g_priv.add((rec_uri, EX.recordDate, Literal(w['date'], datatype=XSD.date)))
    g_priv.add((rec_uri, EX.weightKg, Literal(float(w['weight_kg']), datatype=XSD.decimal)))

private_ttl = g_priv.serialize(format='turtle')
open('private.ttl', 'w', encoding='utf-8').write(private_ttl)
print('✅ Generado: private.ttl')
print(private_ttl[:1200])


## (4) Transformación: datos abiertos (CSV no-RDF) → RDF (Turtle)
Genera `open.ttl` con comunas y regiones, para enlazar geografía de granjas.

In [ ]:
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF

g_open = Graph()
g_open.bind('ex', EX)
g_open.bind('schema', SCHEMA)

comunas = pd.read_csv('comunas_open.csv')

for _, c in comunas.iterrows():
    comuna_uri = URIRef(EX[f"comuna/{int(c['comuna_code'])}"])
    region_uri = URIRef(EX[f"region/{str(c['region_name']).strip().replace(' ', '_')}"])

    g_open.add((comuna_uri, RDF.type, EX.Comuna))
    g_open.add((comuna_uri, SCHEMA.name, Literal(c['comuna_name'])))
    g_open.add((comuna_uri, EX.inRegion, region_uri))

    g_open.add((region_uri, RDF.type, EX.Region))
    g_open.add((region_uri, SCHEMA.name, Literal(c['region_name'])))

open_ttl = g_open.serialize(format='turtle')
open('open.ttl', 'w', encoding='utf-8').write(open_ttl)
print('✅ Generado: open.ttl')
print(open_ttl[:1200])


## (5) Merge de grafos RDF
Genera `merged.ttl` combinando el grafo privado con el grafo de datos abiertos.

In [ ]:
from rdflib import Graph

g = Graph()
g.parse('private.ttl', format='turtle')
g.parse('open.ttl', format='turtle')

merged_ttl = g.serialize(format='turtle')
open('merged.ttl', 'w', encoding='utf-8').write(merged_ttl)
print('✅ Generado: merged.ttl')
print(merged_ttl[:1200])


## (6) Definir ShEx y validar
Se crea `schema.shex` y se valida un `Animal` y un `WeightRecord`.


In [ ]:
from pyshex import ShExEvaluator

schema_shex = '''
PREFIX ex: <https://kg.example.cl/animal/>
PREFIX schema: <https://schema.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

ex:FarmShape {
  a [ex:Farm] ;
  schema:name xsd:string ;
  ex:locatedInComuna IRI
}

ex:AnimalShape {
  a [ex:Animal] ;
  ex:species xsd:string ;
  ex:birthDate xsd:date ;
  ex:raisedInFarm IRI
}

ex:WeightRecordShape {
  a [ex:WeightRecord] ;
  ex:forAnimal IRI ;
  ex:recordDate xsd:date ;
  ex:weightKg xsd:decimal
}

ex:ComunaShape {
  a [ex:Comuna] ;
  schema:name xsd:string ;
  ex:inRegion IRI
}
'''

open('schema.shex', 'w', encoding='utf-8').write(schema_shex)
print('✅ Generado: schema.shex')

# Validar un animal
focus_animal = 'https://kg.example.cl/animal/animal/A001'
shape_animal = 'https://kg.example.cl/animal/AnimalShape'
ev1 = ShExEvaluator(rdf=merged_ttl, schema=schema_shex, focus=focus_animal, start=shape_animal)
r1 = list(ev1.evaluate())[0]
print('Animal A001 conforms:', r1.result)

# Validar un registro de peso
focus_weight = 'https://kg.example.cl/animal/weightRecord/W001'
shape_weight = 'https://kg.example.cl/animal/WeightRecordShape'
ev2 = ShExEvaluator(rdf=merged_ttl, schema=schema_shex, focus=focus_weight, start=shape_weight)
r2 = list(ev2.evaluate())[0]
print('WeightRecord W001 conforms:', r2.result)


## (7) Ejemplo inválido (debe FALLAR)
Se crea `invalid_examples.ttl` con errores de tipo/fecha para demostrar que ShEx detecta inconsistencias.

In [ ]:
invalid_ttl = '''
@prefix ex: <https://kg.example.cl/animal/> .
@prefix schema: <https://schema.org/> .

ex:weightRecord/WBAD a ex:WeightRecord ;
  ex:forAnimal ex:animal/A001 ;
  ex:recordDate "2026-02-30" ;   # ❌ fecha inválida
  ex:weightKg "cincuenta" .      # ❌ debería ser decimal
'''

open('invalid_examples.ttl', 'w', encoding='utf-8').write(invalid_ttl)
print('✅ Generado: invalid_examples.ttl')

# Mezclar merged + invalid y validar WBAD
from rdflib import Graph
g_bad = Graph()
g_bad.parse(data=merged_ttl, format='turtle')
g_bad.parse(data=invalid_ttl, format='turtle')
bad_all = g_bad.serialize(format='turtle')

focus_bad = 'https://kg.example.cl/animal/weightRecord/WBAD'
shape_weight = 'https://kg.example.cl/animal/WeightRecordShape'

ev_bad = ShExEvaluator(rdf=bad_all, schema=schema_shex, focus=focus_bad, start=shape_weight)
rb = list(ev_bad.evaluate())[0]
print('WeightRecord WBAD conforms (debería ser False):', rb.result)


## (8) Descargar outputs
Descarga los archivos generados para subirlos al repo.

In [ ]:
from google.colab import files
for f in ['private.ttl','open.ttl','merged.ttl','schema.shex','invalid_examples.ttl']:
    files.download(f)
